# SDF for quadratic BSpline


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
from typing import Iterable, NamedTuple
from numpy.typing import NDArray

import numpy as np
from numpy.random import random as nprand
import ipywidgets as wg
import k3d

from utils import npvec, arr, arrgs, garr, unflat, f32, disquance, normalize

inf = 100500

In [ ]:
def pladd(plot, *args):
    for a in args:
        plot.__iadd__(a)

# Piece-wisdom


In [ ]:
def controls2_iter(controls: NDArray):
    """Iterate over all possible sets of controls"""
    for i in range(len(controls) - 2):
        yield controls[i : i + 3]

In [ ]:
M2 = np.array([[1, 1, 0], [-2, 2, 0], [1, -2, 1]]) / 2
M2dt = np.array([[-1, 1, 0], [1, -2, 1]])

In [ ]:
def np_powers2(t: NDArray):
    exps = arrgs(0, 1, 2)
    return t[..., None] ** exps[None, ...]


def segm_curve2(controls: NDArray, tspace: NDArray):
    """A segment of curve between the controls, with local tspace 0 <= t <= 1"""
    assert len(controls) == 3
    return np_powers2(tspace) @ M2 @ controls


def mega_curve2(controls: NDArray, tspace: NDArray):
    """All segments of curve"""
    assert len(controls) > 3
    WM = np_powers2(tspace) @ M2
    return np.concat([WM @ ctrl for ctrl in controls2_iter(controls)])

# Bezier Segment

A segment of $B^{(2)}$-Spline between $c_{j-1}, c_{j}, c_{j+1}$ is equivalent to $B^{(2)}$-ezier with $c_1 = ½(c_{j-1} + c_j), c_2 = c_j, c_3 = ⅓(c_{j+1} + c_j)$

Centering around $c2$ and $t \in [-0.5, +0.5]$

- $c_2 \to 0$
- $c_1 \to v_1 = c_1 - c_2$
- $c_3 \to v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axis
- $v_d = v_3 - v_1$ — parabolic direction
- $c_a = S(0) - c_2 = .25 v_a$ ­— apex of the parabola (closest to $c_2$, max curvature)
- $p_s = p - c_2$
- $p_c = c_a - p_s$ ­— sample relative to apex

$$
S(t') - c_2 =
\big[1, t', t'^2 \big]
\begin{bmatrix}
c_a \\
v_d \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t' \big]
\begin{bmatrix}
v_d \\
2 v_a \\
\end{bmatrix}
$$

### Projecting

$$
[1, t, t^2, t^3]
\begin{pmatrix}
p_c·v_d \\
2 p_c·v_a + v_d·v_d\\
3 v_a·v_d \\
2 v_a·v_a \\
\end{pmatrix}
=0
\tag{perpendicularity}
$$

Assuming the segment is nearly flat.

Solving in $XY$ 2D coords, restoring $z$ for projection.

### Orientation

- $O$ — vector from projection to sample
- $T$ — tangential vector at projection point
- $[O×T]_z = O_x T_y - O_y T_x$

$$
\begin{cases}
[O×T]_z > 0 \implies \text{O points to right-hand side} \\
[O×T]_z < 0 \implies \text{O points to left-hand side} \\
\tag{orientation}
\end{cases}
$$

### Horizon

$$
\begin{cases}
p_s · v_1 > v_1 · v_1 \implies t'_{prj} < -0.5
\\
p_s · v_3 > v_3 · v_3 \implies t'_{prj} > +0.5
\end{cases}
\tag{visibility}
$$


In [ ]:
from typing import Self


class Bezielt:
    """Centered at c_2 and t=-0.5..+0.5"""

    c2: npvec
    ca: npvec
    v1: npvec
    v3: npvec
    va: npvec
    vd: npvec

    @classmethod
    def BSegm(cls, bcontrols: NDArray) -> Self:
        p1 = bcontrols[0]
        p2 = bcontrols[1]
        p3 = bcontrols[2]
        return cls((p1 + p2) * 0.5, p2, (p3 + p2) * 0.5)

    def __init__(self, c1: npvec, c2: npvec, c3: npvec):
        self.c2 = c2
        self.v1 = c1 - c2
        self.v3 = c3 - c2
        self.va = self.v3 + self.v1
        self.vd = self.v3 - self.v1
        self.ca = 0.25 * self.va

    def __repr__(self):
        return f"Bezielt({self.c2 + self.v1}, {self.c2}, {self.c2 + self.v3})"

    def loc(self, point: npvec | NDArray) -> npvec:
        return point - self.c2

    def glb(self, point: npvec | NDArray) -> npvec:
        return point + self.c2

    def point(self, t: float) -> npvec:
        return t * t * self.va + t * self.vd + self.ca

    def flow(self, t: float) -> npvec:
        return 2 * self.va * t + self.vd

    def curve(self, tspace: NDArray) -> NDArray:
        return garr(self.point(t) for t in tspace)

In [ ]:
from math import sqrt, cbrt, cos, acos, pi


def solve(bezier: Bezielt, p: npvec) -> tuple[float, ...]:
    va = bezier.va[:2]
    vd = bezier.vd[:2]
    pc = bezier.ca[:2] - p[:2]

    aa = float(va @ va)
    ad = float(va @ vd)
    dd = float(vd @ vd)
    pa = float(pc @ va)
    pd = float(pc @ vd)

    a_1 = dd + 2 * pa
    p3 = (2 * aa * a_1 - 3 * ad**2) / (12 * aa**2)
    q2 = (ad**3 - aa * ad * a_1 + 2 * aa**2 * pd) / (8 * aa**3)
    off = ad / (2 * aa)

    D = p3**3 + q2**2

    if D > 0:
        C = cbrt(sqrt(D) - q2)
        t_ = C - p3 / C
        return (t_ - off,)
    else:
        rt = sqrt(-p3)
        k = 2 * rt
        ph = 2 * pi / 3
        th = acos(q2 / (p3 * rt)) / 3
        t_0 = k * cos(th)
        t_1 = k * cos(th - ph)
        t_2 = k * cos(th - 2 * ph)
        return (t_0 - off, t_1 - off, t_2 - off)

In [ ]:
def side(bezier: Bezielt, t: float, p: npvec) -> int:
    """+1 for left hand side, -1 for right-hand side"""
    ort = p - bezier.point(t)
    tng = bezier.flow(t)
    crz = ort[1] * tng[0] - ort[0] * tng[1]
    return int(np.sign(crz))

In [ ]:
def horizon(bezier: Bezielt, sample: npvec) -> int:
    """number of edges in potential reachability, 0 = OOB"""
    p = bezier.loc(sample)
    h1 = bezier.v1 @ bezier.v1
    h3 = bezier.v3 @ bezier.v3
    p1 = bezier.v1 @ p
    p3 = bezier.v3 @ p
    return int(p1 < h1) + int(p3 < h3)

In [ ]:
class Projection(NamedTuple):
    smp: npvec
    t: float
    pnt: npvec
    dsq: float = 0
    sgn: int = 0
    j: int = 0

    @property
    def sdist(self):
        return self.sgn * np.sqrt(self.dsq)

In [ ]:
def project(bezier: Bezielt, sample: npvec) -> Projection:
    ps = bezier.loc(sample)
    tt = solve(bezier, ps)
    tt = tuple(filter(lambda t: -0.5 <= t <= 0.5, tt))

    if len(tt) == 0:
        return Projection(sample, 0, sample, inf, 0)

    if len(tt) == 1:
        t = tt[0]
        pnt = bezier.point(t)
        dsq = disquance(pnt, ps)
    else:
        pnts = tuple(bezier.point(t) for t in tt)
        dsqs = tuple(disquance(p, ps) for p in pnts)
        best = np.argmin(dsqs)
        t = tt[best]
        pnt = pnts[best]
        dsq = dsqs[best]

    sgn = side(bezier, t, ps)

    return Projection(sample, t, bezier.glb(pnt), dsq, sgn)


In [ ]:
def projectall(segments: Iterable[Bezielt], sample: npvec) -> Projection:
    projs = [project(segm, sample) for segm in segments]

    if len(projs) == 0:
        return Projection(sample, 0, sample, inf, 0, 0)

    dsqs = [prj.dsq for prj in projs]
    best = int(np.argmin(dsqs))
    proj = projs[best]
    return Projection(proj.smp, proj.t, proj.pnt, proj.dsq, proj.sgn, best + 1)

In [ ]:
def projectviz(segments: Iterable[Bezielt], sample: npvec) -> Projection:
    projs = [project(segm, sample) for segm in segments if horizon(segm, sample)]

    if len(projs) == 0:
        return Projection(sample, 0, sample, inf, 0, 0)

    dsqs = [prj.dsq for prj in projs]
    best = int(np.argmin(dsqs))
    proj = projs[best]
    return Projection(proj.smp, proj.t, proj.pnt, proj.dsq, proj.sgn, best + 1)

---


In [ ]:
def random_points(n: int):
    l = np.linspace(-0.75, 0.75, n)
    return f32(arrgs(l, l, np.zeros(n)).T + (nprand((n, 3)) - 0.5) * arrgs(1.0, 1.0, 0.125))

In [ ]:
N = 7
controls = random_points(N)

bezier = Bezielt.BSegm(controls)
tspace = np.linspace(-0.5, +0.5, 16, dtype=np.float32)
tspace[0] += 0.001
tspace[-1] -= 0.001

In [ ]:
plot = k3d.Plot(
    height=720,
    background_color=0x404040,
    grid_color=0x383838,
    label_color=0x000000,
    menu_visibility=False,
    grid=(-1.0, -1.0, 0.0, 1.0, 1.0, 0.5),
    grid_auto_fit=False,
    mode="callback",
)
plot.layout = wg.Layout(width="720px", height="720px")
# plot.camera_auto_fit = False
# plot.camera = [0, 0, 5, 0, 1, 0, 0, 0, 0]

In [ ]:
randomize_btn = wg.Button(description="randomize")
sample_btn = wg.Button(description="sample")
toggle0 = wg.Checkbox(description="controls", value=False)
toggle1 = wg.Checkbox(description="tangents", value=False)
toggle2 = wg.Checkbox(description="texture", value=False)
layer_sw = wg.RadioButtons(description="layers", options=["", "sdf", "t", "idx", "hrz", "viz"], value="")

In [ ]:
wg.HBox([plot, wg.VBox([toggle0, toggle1, toggle2, layer_sw, randomize_btn, sample_btn])], layout=dict(width="100%", grid_gap="8px"))

In [ ]:
k3controls = k3d.line(vertices=controls, color=0x808080, line_width=0.125, shader="thick", visible=toggle0.value)
k3curve = k3d.line(vertices=[], shader="mesh", color=0xF0F0F0, line_width=0.25, color_map=k3d.matplotlib_color_maps.Rainbow, color_range=[-0.5, +0.5])
k3flow = k3d.vectors(origins=[(0, 0, 0)], vectors=[(0, 0, 0)], use_head=False, color=0x000000, head_color=0xF0F0F0, visible=False)
k3points = k3d.points(positions=[], shader="mesh", point_size=0.03125, color_map=k3d.matplotlib_color_maps.Seismic, color_range=[-0.5, +0.5])
k3proj = k3d.line(vertices=[], attribute=[], shader="thick", line_width=0.125, color_map=k3d.matplotlib_color_maps.Seismic, color_range=[-0.5, +0.5])


pladd(plot, k3proj, k3points, k3controls, k3curve, k3flow)

In [ ]:
def toggle(obj, val):
    obj.visible = val


toggle0.observe(lambda ch: toggle(k3controls, ch.new), "value")
toggle1.observe(lambda ch: toggle(k3flow, ch.new), "value")

In [ ]:
def regenerate_curve():
    segments = [Bezielt.BSegm(ctrl) for ctrl in controls2_iter(controls)]
    k3curve.vertices = np.concat([bezier.glb(bezier.curve(tspace)) for bezier in segments])
    k3curve.attribute = np.tile(tspace, len(segments))
    k3flow.origins = garr(bezier.glb(bezier.point(0.0)) for bezier in segments)
    k3flow.vectors = garr(normalize(bezier.flow(0.0)) for bezier in segments) * 0.25
    k3flow.visible = toggle1.value


regenerate_curve()

In [ ]:
def randomize():
    global bezier
    controls[:] = random_points(N)
    k3controls.vertices = f32(controls)
    regenerate_curve()
    regenerate_image(layer_sw.value)
    k3proj.vertices = []
    k3proj.attribute = []
    k3points.positions = []
    k3points.attribute = []

In [ ]:
randomize_btn.on_click(lambda _: randomize())

In [ ]:
def sample_random():
    segments = [Bezielt.BSegm(ctrl) for ctrl in controls2_iter(controls)]
    sample = unflat(nprand(2) - 0.5)
    proj = projectall(segments, sample)
    k3proj.vertices = [sample, proj.pnt]
    k3proj.attribute = [proj.sdist, proj.sdist]
    k3points.positions = [sample, proj.pnt]
    k3points.attribute = [proj.sdist, proj.sdist]

In [ ]:
sample_btn.on_click(lambda btn: sample_random())

## Texture


In [ ]:
RES = 64
PIX = 1.0 / RES
X, Y = np.meshgrid(np.arange(-0.5, 0.5, PIX), np.arange(-0.5, 0.5, PIX))
COORDS = np.stack((Y, X)).T.reshape((RES * RES, 2)) + 0.5 * PIX  # pixel centers

In [ ]:
imagedata = np.zeros((RES, RES))
k3image = k3d.texture(attribute=imagedata, interpolation=False, color_map=k3d.colormaps.basic_color_maps.Binary, color_range=[-1.0, 1.0])
pladd(plot, k3image)
plot.colorbar_object_id = k3image.id

In [ ]:
toggle(k3image, toggle2.value)
toggle2.observe(lambda ch: toggle(k3image, ch.new), "value")

In [ ]:
def regenerate_image(layer: str):
    global imagedata
    segments = [Bezielt.BSegm(ctrl) for ctrl in controls2_iter(controls)]
    if layer == "sdf":
        imagedata = garr(projectviz(segments, unflat(s)).sdist for s in COORDS)
        k3image.attribute = imagedata.reshape((RES, RES))
        k3image.color_map = k3d.matplotlib_color_maps.Seismic
        k3image.color_range = [-1, +1]
    elif layer == "t":
        imagedata = garr(projectviz(segments, unflat(s)).t for s in COORDS)
        k3image.attribute = imagedata.reshape((RES, RES))
        k3image.color_map = k3d.basic_color_maps.Rainbow
        k3image.color_range = [-0.5, +0.5]
    elif layer == "idx":
        imagedata = garr(projectall(segments, unflat(s)).j for s in COORDS)
        k3image.attribute = imagedata.reshape((RES, RES))
        k3image.color_map = k3d.matplotlib_color_maps.Tab10
        k3image.color_range = [0, 9]
    elif layer == "hrz":

        def hrz(sample):
            return sum(horizon(segm, sample) for segm in segments)

        imagedata = garr(hrz(unflat(s)) for s in COORDS)
        k3image.attribute = imagedata.reshape((RES, RES))
        k3image.color_map = k3d.matplotlib_color_maps.Binary_r
        k3image.color_range = [0, N * 2]
    elif layer == "viz":

        def viz(sample):
            return sum(horizon(segm, sample) > 0 for segm in segments)

        imagedata = garr(viz(unflat(s)) for s in COORDS)
        k3image.attribute = imagedata.reshape((RES, RES))
        k3image.color_map = k3d.matplotlib_color_maps.Binary_r
        k3image.color_range = [0, N]
    else:
        imagedata = nprand((RES, RES)) - 0.5
        k3image.color_map = k3d.matplotlib_color_maps.Binary
        k3image.color_range = [0, 1]


In [ ]:
layer_sw.observe(lambda ch: regenerate_image(ch.new), "value")